In [ ]:
from parser import Parser
import shapely
from shapely.ops import transform
import osmnx as ox
import geopandas as gpd
import pathlib
import os


import networkx as nx
import numpy as np
from shapely.geometry import LineString, Point
import math
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

import numpy.typing as npt

from scipy.spatial import cKDTree
from shapely.strtree import STRtree

from shapely.geometry import LineString
import math

from utils.nx import snap_to_edges, build_final_path, filter_edges

In [ ]:
save_dir = pathlib.Path("/mnt/c/Users/nikita/qgisData/busroutes_new")
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
CITY_NAME = "Санкт-Петербург"
MERCATOR = 3857
WGS_84 = 4326
R_CONSOLIDATE_M = 12.0  # radius to fuse multi-node junctions (in meters)


bus_parser = Parser.BusGraphParser(CITY_NAME)
tram_parser = Parser.TramGraphParser(CITY_NAME)
trolleybus_parser = Parser.TrolleyGraphParser(CITY_NAME)

all_routes = []

for route_info in trolleybus_parser.get_all_routes_info():
    route_name = route_info[0]
    route_url = route_info[2]
    all_routes.append((route_name, route_url))
for route_info in bus_parser.get_all_routes_info():
    route_name = route_info[0]
    route_url = route_info[2]
    all_routes.append((route_name, route_url))
for route_info in tram_parser.get_all_routes_info():
    route_name = route_info[0]
    route_url = route_info[2]
    all_routes.append((route_name, route_url))

    

In [ ]:
for (route_name, route_url) in all_routes:
    print(route_name)
    route_list = bus_parser.get_route(route_url)

In [ ]:
with open("input/boundaries.geojson", "r") as f:
    geojson = f.read()
boundaries = shapely.from_geojson(geojson)
graph_raw = ox.graph_from_polygon(boundaries, network_type="drive", simplify=False)
graph_merc = ox.project_graph(graph_raw, to_crs=MERCATOR)
graph = ox.consolidate_intersections(graph_merc, tolerance=R_CONSOLIDATE_M, rebuild_graph=True)

nodes, edges = ox.graph_to_gdfs(graph)





In [ ]:
geoms = edges["geometry"]
rtree = STRtree(edges["geometry"])
edges["routes"] = 0

In [ ]:
with open(save_dir / "edges.geojson", "w") as f:
    f.write(edges.to_json())